# 🏆 XAUUSD ML Analysis — High Accuracy Trading Signal

**Stack:** LightGBM · XGBoost · RandomForest · Stacking · LSTM  
**Data:** XAUUSD 5m OHLCV  
**Target:** Prediksi arah 3-candle ke depan dengan akurasi >80%

---
### 📋 Notebook Flow
1. **Data Loading & EDA** — eksplorasi distribusi, statistik, seasonality  
2. **Feature Engineering** — 120+ fitur teknikal dari indicators.py  
3. **Label Generation** — multi-threshold, forward-return label  
4. **Preprocessing** — scaling, feature selection, class balance  
5. **Baseline Models** — LogReg, DecisionTree benchmark  
6. **Ensemble Models** — LGB + XGB + RF + ET + Stacking  
7. **Hyperparameter Tuning** — Optuna Bayesian search  
8. **Walk-Forward Validation** — time-series CV  
9. **Deep Learning** — CNN-BiLSTM  
10. **SHAP Feature Importance**  
11. **Signal Backtest** — P&L, drawdown, win rate  
12. **Model Export** — simpan model terbaik

In [ ]:
# ── 0. Install dependencies (jalankan sekali) ──────────────────────────────
import subprocess, sys

pkgs = [
    'lightgbm', 'xgboost', 'scikit-learn', 'optuna',
    'shap', 'matplotlib', 'seaborn', 'pandas', 'numpy',
    'joblib', 'plotly'
]

for p in pkgs:
    try:
        __import__(p.replace('-', '_'))
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', p, '-q'])

print('✅ Semua dependencies OK')

In [ ]:
# ── 1. Imports ─────────────────────────────────────────────────────────────
import sys, os, warnings, json
from pathlib import Path

warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from sklearn.preprocessing import RobustScaler
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    ExtraTreesClassifier, VotingClassifier, StackingClassifier
)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve
)
import lightgbm as lgb
import xgboost as xgb
import joblib
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Path setup ─────────────────────────────────────────────────────────────
ROOT = Path('..').resolve().parent   # trader-ai root
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

plt.style.use('dark_background')
sns.set_palette('husl')
pd.set_option('display.max_columns', 50)

SEED = 42
np.random.seed(SEED)

print(f'ROOT = {ROOT}')
print(f'pandas {pd.__version__} | numpy {np.__version__} | lgb {lgb.__version__} | xgb {xgb.__version__}')

---
## 📊 Section 1 — Data Loading & EDA

In [ ]:
# ── 1.1 Load OHLCV ─────────────────────────────────────────────────────────
DATA_PATH = ROOT / 'data' / 'history' / 'XAUUSD_5m.csv'
assert DATA_PATH.exists(), f'File tidak ditemukan: {DATA_PATH}'

df_raw = pd.read_csv(DATA_PATH, index_col=0, parse_dates=True)
df_raw.columns = [c.capitalize() for c in df_raw.columns]
df_raw.index.name = 'Datetime'
df_raw = df_raw[['Open','High','Low','Close','Volume']].dropna()
df_raw = df_raw.sort_index()

print(f'Total candles : {len(df_raw):,}')
print(f'Date range    : {df_raw.index[0]} → {df_raw.index[-1]}')
print(f'NaN count     : {df_raw.isna().sum().sum()}')
df_raw.tail(3)

In [ ]:
# ── 1.2 Basic Statistics ────────────────────────────────────────────────────
df_raw.describe().round(2)

In [ ]:
# ── 1.3 Return Distribution ─────────────────────────────────────────────────
returns = df_raw['Close'].pct_change().dropna() * 100

fig, axes = plt.subplots(2, 3, figsize=(18, 9))

# Price
axes[0,0].plot(df_raw['Close'], lw=0.5, color='#00d4ff')
axes[0,0].set_title('XAUUSD Close Price')
axes[0,0].set_ylabel('Price (USD)')

# Returns histogram
axes[0,1].hist(returns, bins=100, color='#ff6b35', edgecolor='none', alpha=0.8)
axes[0,1].axvline(returns.mean(), color='white', lw=1, label=f'Mean: {returns.mean():.3f}%')
axes[0,1].set_title('Return Distribution (%)')
axes[0,1].legend()

# Volume
axes[0,2].bar(df_raw.index, df_raw['Volume'], color='#a8edea', alpha=0.6, width=0.003)
axes[0,2].set_title('Volume')

# Hourly return heatmap
df_raw['hour'] = df_raw.index.hour
df_raw['dow'] = df_raw.index.dayofweek
df_raw['ret'] = df_raw['Close'].pct_change()
pivot = df_raw.pivot_table('ret', 'hour', 'dow', aggfunc='mean') * 100
pivot.columns = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun'][:len(pivot.columns)]
sns.heatmap(pivot, ax=axes[1,0], cmap='RdYlGn', center=0, fmt='.3f', annot=True, annot_kws={'size':6})
axes[1,0].set_title('Avg Return by Hour & Day (%)')

# ATR over time
high_low = df_raw['High'] - df_raw['Low']
atr_simple = high_low.rolling(14).mean()
axes[1,1].plot(atr_simple, lw=0.5, color='#c3f0ca')
axes[1,1].set_title('ATR(14) Over Time')

# Rolling Volatility
vol = returns.rolling(50).std()
axes[1,2].plot(vol, lw=0.5, color='#f7971e')
axes[1,2].set_title('Rolling 50-candle Volatility')

plt.tight_layout()
plt.savefig('eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\nReturn skew: {returns.skew():.3f} | kurtosis: {returns.kurtosis():.3f}')

In [ ]:
# ── 1.4 Session Analysis ────────────────────────────────────────────────────
def get_session(hour):
    if 0 <= hour < 7:   return 'Asia'
    if 7 <= hour < 12:  return 'London'
    if 12 <= hour < 20: return 'New York'
    return 'Off'

df_raw['session'] = df_raw['hour'].apply(get_session)
session_stats = df_raw.groupby('session').agg(
    avg_return=('ret','mean'),
    std_return=('ret','std'),
    avg_volume=('Volume','mean'),
    candle_count=('Close','count')
).round(6)

session_stats['avg_return_pct'] = session_stats['avg_return'] * 100
session_stats['volatility_pct'] = session_stats['std_return'] * 100
print('\n📅 Session Analysis:')
print(session_stats[['avg_return_pct','volatility_pct','avg_volume','candle_count']].to_string())

---
## ⚙️ Section 2 — Feature Engineering

In [ ]:
# ── 2.1 Load existing indicators ────────────────────────────────────────────
try:
    from ai.indicators import add_all_indicators
    df = df_raw.copy()
    df = add_all_indicators(df, max_rows=0)   # 0 = all rows
    print(f'✅ indicators.py loaded. Columns: {len(df.columns)}')
except Exception as e:
    print(f'⚠️  indicators.py error: {e}')
    print('   Menghitung ulang manual...')
    df = df_raw.copy()

In [ ]:
# ── 2.2 Extra Feature Engineering ──────────────────────────────────────────
def add_extra_features(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    c = d['Close']; h = d['High']; l = d['Low']; o = d['Open']; v = d['Volume']

    # ── Candle body features
    d['body_size']      = abs(c - o)
    d['upper_wick']     = h - d[['Open','Close']].max(axis=1)
    d['lower_wick']     = d[['Open','Close']].min(axis=1) - l
    d['candle_range']   = h - l
    d['body_pct']       = d['body_size'] / (d['candle_range'] + 1e-9)
    d['is_bullish']     = (c > o).astype(int)

    # ── Multi-period returns
    for n in [1, 2, 3, 5, 8, 13, 21]:
        d[f'ret_{n}']  = c.pct_change(n)
        d[f'high_{n}'] = h.rolling(n).max()
        d[f'low_{n}']  = l.rolling(n).min()

    # ── Price position relative to range
    for n in [5, 10, 20, 50]:
        rng = d[f'high_{n}'] - d[f'low_{n}']
        d[f'pos_in_range_{n}'] = (c - d[f'low_{n}']) / (rng + 1e-9)

    # ── RSI (if not in df)
    if 'rsi' not in d.columns:
        delta = c.diff()
        gain  = delta.clip(lower=0).rolling(14).mean()
        loss  = (-delta.clip(upper=0)).rolling(14).mean()
        d['rsi'] = 100 - (100 / (1 + gain / (loss + 1e-9)))

    # ── MACD (if not in df)
    if 'macd' not in d.columns:
        ema12 = c.ewm(span=12, adjust=False).mean()
        ema26 = c.ewm(span=26, adjust=False).mean()
        d['macd']        = ema12 - ema26
        d['macd_signal'] = d['macd'].ewm(span=9, adjust=False).mean()
        d['macd_hist']   = d['macd'] - d['macd_signal']

    # ── EMAs (if not in df)
    for p in [9, 20, 50, 100, 200]:
        col = f'ema_{p}'
        if col not in d.columns:
            d[col] = c.ewm(span=p, adjust=False).mean()
        d[f'price_vs_ema{p}'] = (c - d[col]) / (d[col] + 1e-9)

    # ── ATR (if not in df)
    if 'atr' not in d.columns:
        tr = pd.concat([
            h - l,
            (h - c.shift()).abs(),
            (l - c.shift()).abs()
        ], axis=1).max(axis=1)
        d['atr'] = tr.rolling(14).mean()

    # ── Normalised body/range by ATR
    d['body_atr_ratio']  = d['body_size']    / (d['atr'] + 1e-9)
    d['range_atr_ratio'] = d['candle_range'] / (d['atr'] + 1e-9)

    # ── Volume features
    d['vol_ratio_20']  = v / (v.rolling(20).mean() + 1e-9)
    d['vol_ratio_5']   = v / (v.rolling(5).mean()  + 1e-9)
    d['vol_ma20']      = v.rolling(20).mean()

    # ── Consecutive candles
    d['bull_streak'] = (
        (c > o).astype(int)
        .groupby((c > o).ne((c > o).shift()).cumsum())
        .cumcount() + 1
    ) * (c > o).astype(int)

    # ── Time features
    d['hour']          = d.index.hour
    d['day_of_week']   = d.index.dayofweek
    d['is_london']     = ((d['hour'] >= 7) & (d['hour'] < 12)).astype(int)
    d['is_ny']         = ((d['hour'] >= 12) & (d['hour'] < 20)).astype(int)

    # ── Lag features of RSI, MACD
    for lag in [1, 2, 3]:
        d[f'rsi_lag{lag}']       = d['rsi'].shift(lag)
        d[f'macd_hist_lag{lag}'] = d['macd_hist'].shift(lag)

    return d

df = add_extra_features(df)
print(f'Total features after engineering: {len(df.columns)}')

---
## 🏷️ Section 3 — Label Generation

In [ ]:
# ── 3.1 Forward Return Label ────────────────────────────────────────────────
LOOKAHEAD       = 3        # candles ke depan
THRESHOLD_PCT   = 0.0008   # 0.08% = threshold BUY/SELL vs SIDEWAYS

def make_labels(df: pd.DataFrame, lookahead=3, threshold=0.0008):
    """1=BUY, 0=SELL, NaN=sideways (dihapus)"""
    fwd_ret = df['Close'].shift(-lookahead) / df['Close'] - 1
    label = pd.Series(np.nan, index=df.index)
    label[fwd_ret >  threshold] = 1
    label[fwd_ret < -threshold] = 0
    return label

df['label'] = make_labels(df, LOOKAHEAD, THRESHOLD_PCT)

label_counts = df['label'].value_counts()
sideways     = df['label'].isna().sum()
total_labeled = label_counts.sum()

print(f'Label distribution:')
print(f'  BUY      : {int(label_counts.get(1,0)):,} ({label_counts.get(1,0)/total_labeled*100:.1f}%)')
print(f'  SELL     : {int(label_counts.get(0,0)):,} ({label_counts.get(0,0)/total_labeled*100:.1f}%)')
print(f'  Sideways : {sideways:,} (removed)')
print(f'  Total    : {total_labeled:,} usable labels')

fig, ax = plt.subplots(figsize=(6,4))
ax.bar(['BUY','SELL','Sideways'], [label_counts.get(1,0), label_counts.get(0,0), sideways],
       color=['#00ff88','#ff4466','#888888'])
ax.set_title('Label Distribution')
plt.tight_layout()
plt.show()

---
## 🔧 Section 4 — Preprocessing & Feature Selection

In [ ]:
# ── 4.1 Prepare X, y ────────────────────────────────────────────────────────
EXCLUDE_COLS = [
    'label','Open','High','Low','Close','Volume',
    'session','ret','hour','dow'
]

df_clean = df.dropna(subset=['label']).copy()

feature_cols = [c for c in df_clean.columns
                if c not in EXCLUDE_COLS
                and df_clean[c].dtype in [np.float64, np.float32, np.int64, np.int32, int, float]]

X_all = df_clean[feature_cols]
y_all = df_clean['label'].astype(int)

# Drop columns with >30% NaN
null_pct = X_all.isnull().mean()
bad_cols = null_pct[null_pct > 0.30].index.tolist()
X_all = X_all.drop(columns=bad_cols)
print(f'Dropped {len(bad_cols)} columns with >30% NaN: {bad_cols[:5]}...')

# Fill remaining NaN with forward fill then 0
X_all = X_all.ffill().fillna(0)

# Drop inf
X_all = X_all.replace([np.inf, -np.inf], 0)

print(f'\nFeature matrix: {X_all.shape}')
print(f'Label shape   : {y_all.shape}')
print(f'Class balance : {y_all.mean():.3f} (1=BUY)')

In [ ]:
# ── 4.2 Train/Test Split (walk-forward safe: chronological 80/20) ───────────
SPLIT = 0.80
n     = len(X_all)
split_idx = int(n * SPLIT)

X_train, X_test = X_all.iloc[:split_idx], X_all.iloc[split_idx:]
y_train, y_test = y_all.iloc[:split_idx], y_all.iloc[split_idx:]

# ── 4.3 Scaling ─────────────────────────────────────────────────────────────
scaler  = RobustScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# ── 4.4 Feature Selection (Top K by mutual info) ─────────────────────────────
K_FEATURES = 70
selector   = SelectKBest(mutual_info_classif, k=K_FEATURES)
X_train_sel = selector.fit_transform(X_train_s, y_train)
X_test_sel  = selector.transform(X_test_s)

selected_features = np.array(X_train.columns)[selector.get_support()].tolist()

print(f'Train : {X_train_sel.shape} | Test : {X_test_sel.shape}')
print(f'Top 10 features:')
scores = pd.Series(selector.scores_[selector.get_support()], index=selected_features)
print(scores.nlargest(10).to_string())

In [ ]:
# ── 4.5 Feature Correlation Heatmap (top 20) ────────────────────────────────
top20 = scores.nlargest(20).index.tolist()
corr  = X_train[top20].corr()

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, annot_kws={'size':7})
ax.set_title('Top-20 Feature Correlation Matrix')
plt.tight_layout()
plt.savefig('feature_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 🤖 Section 5 — Baseline Models

In [ ]:
# ── 5.1 Helper: evaluate model ──────────────────────────────────────────────
RESULTS = {}

def eval_model(name, model, X_tr, y_tr, X_te, y_te, threshold=0.5):
    model.fit(X_tr, y_tr)
    if hasattr(model, 'predict_proba'):
        prob = model.predict_proba(X_te)[:, 1]
        pred = (prob >= threshold).astype(int)
        auc  = roc_auc_score(y_te, prob)
    else:
        pred = model.predict(X_te)
        prob = None
        auc  = 0.0

    acc  = accuracy_score(y_te, pred)
    prec = precision_score(y_te, pred, zero_division=0)
    rec  = recall_score(y_te, pred, zero_division=0)
    f1   = f1_score(y_te, pred, zero_division=0)

    # Confident accuracy (top 30% confidence)
    if prob is not None:
        conf_mask = (prob >= 0.70) | (prob <= 0.30)
        conf_acc  = accuracy_score(y_te[conf_mask], pred[conf_mask]) if conf_mask.sum() > 0 else 0
        conf_cov  = conf_mask.mean() * 100
    else:
        conf_acc = conf_cov = 0

    RESULTS[name] = {
        'acc': acc, 'prec': prec, 'rec': rec, 'f1': f1,
        'auc': auc, 'conf_acc': conf_acc, 'conf_cov': conf_cov
    }

    print(f'[{name}]')
    print(f'  Acc={acc:.4f}  Prec={prec:.4f}  Rec={rec:.4f}  F1={f1:.4f}')
    print(f'  AUC={auc:.4f}  ConfAcc={conf_acc:.4f}  ConfCoverage={conf_cov:.1f}%')
    return model

# ── 5.2 Baseline ─────────────────────────────────────────────────────────────
lr  = eval_model('LogReg', LogisticRegression(max_iter=500, C=0.1, random_state=SEED),
                 X_train_sel, y_train, X_test_sel, y_test)

dt  = eval_model('DecisionTree', DecisionTreeClassifier(max_depth=6, random_state=SEED),
                 X_train_sel, y_train, X_test_sel, y_test)

---
## 🚀 Section 6 — Ensemble Models

In [ ]:
# ── 6.1 Default Ensemble ─────────────────────────────────────────────────────
lgbm_base = lgb.LGBMClassifier(
    n_estimators=500, learning_rate=0.05, max_depth=6,
    num_leaves=31, subsample=0.8, colsample_bytree=0.8,
    min_child_samples=20, random_state=SEED, verbose=-1
)

xgb_base = xgb.XGBClassifier(
    n_estimators=500, learning_rate=0.05, max_depth=5,
    subsample=0.8, colsample_bytree=0.8, eval_metric='logloss',
    use_label_encoder=False, random_state=SEED, verbosity=0
)

rf_base = RandomForestClassifier(
    n_estimators=300, max_depth=8, min_samples_leaf=10,
    n_jobs=-1, random_state=SEED
)

et_base = ExtraTreesClassifier(
    n_estimators=300, max_depth=8, min_samples_leaf=10,
    n_jobs=-1, random_state=SEED
)

gb_base = GradientBoostingClassifier(
    n_estimators=300, learning_rate=0.05, max_depth=4,
    subsample=0.8, random_state=SEED
)

lgbm_m = eval_model('LightGBM', lgbm_base, X_train_sel, y_train, X_test_sel, y_test)
xgb_m  = eval_model('XGBoost',  xgb_base,  X_train_sel, y_train, X_test_sel, y_test)
rf_m   = eval_model('RandomForest', rf_base, X_train_sel, y_train, X_test_sel, y_test)
et_m   = eval_model('ExtraTrees', et_base, X_train_sel, y_train, X_test_sel, y_test)

In [ ]:
# ── 6.2 Voting Classifier ────────────────────────────────────────────────────
voting = VotingClassifier(
    estimators=[
        ('lgb', lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                                    num_leaves=31, random_state=SEED, verbose=-1)),
        ('xgb', xgb.XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=5,
                                   eval_metric='logloss', use_label_encoder=False,
                                   random_state=SEED, verbosity=0)),
        ('rf',  RandomForestClassifier(n_estimators=300, max_depth=8,
                                        min_samples_leaf=10, n_jobs=-1, random_state=SEED))
    ],
    voting='soft', weights=[2, 2, 1]
)

voting_m = eval_model('Voting(LGB+XGB+RF)', voting,
                      X_train_sel, y_train, X_test_sel, y_test)

In [ ]:
# ── 6.3 Stacking Classifier ──────────────────────────────────────────────────
stacking = StackingClassifier(
    estimators=[
        ('lgb', lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05,
                                    random_state=SEED, verbose=-1)),
        ('xgb', xgb.XGBClassifier(n_estimators=300, learning_rate=0.05,
                                   eval_metric='logloss', use_label_encoder=False,
                                   random_state=SEED, verbosity=0)),
        ('et',  ExtraTreesClassifier(n_estimators=200, max_depth=7,
                                      n_jobs=-1, random_state=SEED))
    ],
    final_estimator=CalibratedClassifierCV(
        LogisticRegression(C=1.0, max_iter=300), method='isotonic'
    ),
    cv=3, stack_method='predict_proba', n_jobs=-1
)

stacking_m = eval_model('Stacking(LGB+XGB+ET→LR)', stacking,
                        X_train_sel, y_train, X_test_sel, y_test)

---
## 🎯 Section 7 — Hyperparameter Tuning (Optuna)

In [ ]:
# ── 7.1 LightGBM Optuna Tuning ───────────────────────────────────────────────
import optuna
from sklearn.model_selection import TimeSeriesSplit

N_TRIALS   = 50   # Naikkan ke 100+ untuk hasil lebih baik
N_CV_FOLDS = 5
tscv       = TimeSeriesSplit(n_splits=N_CV_FOLDS)

def lgb_objective(trial):
    params = {
        'n_estimators'      : trial.suggest_int('n_estimators', 200, 1000),
        'learning_rate'     : trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'max_depth'         : trial.suggest_int('max_depth', 3, 10),
        'num_leaves'        : trial.suggest_int('num_leaves', 15, 127),
        'subsample'         : trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree'  : trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_samples' : trial.suggest_int('min_child_samples', 10, 100),
        'reg_alpha'         : trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda'        : trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'random_state'      : SEED,
        'verbose'           : -1,
        'n_jobs'            : -1
    }
    model = lgb.LGBMClassifier(**params)
    scores_ = cross_val_score(model, X_train_sel, y_train,
                               cv=tscv, scoring='f1', n_jobs=-1)
    return scores_.mean()

study_lgb = optuna.create_study(direction='maximize',
                                 sampler=optuna.samplers.TPESampler(seed=SEED))
study_lgb.optimize(lgb_objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f'\n✅ Best LGB params (F1={study_lgb.best_value:.4f}):')
print(study_lgb.best_params)

In [ ]:
# ── 7.2 XGBoost Optuna Tuning ────────────────────────────────────────────────
def xgb_objective(trial):
    params = {
        'n_estimators'    : trial.suggest_int('n_estimators', 200, 1000),
        'learning_rate'   : trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'max_depth'       : trial.suggest_int('max_depth', 3, 10),
        'subsample'       : trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
        'gamma'           : trial.suggest_float('gamma', 0, 5),
        'reg_alpha'       : trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda'      : trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'eval_metric'     : 'logloss',
        'use_label_encoder': False,
        'random_state'    : SEED,
        'verbosity'       : 0,
        'n_jobs'          : -1
    }
    model = xgb.XGBClassifier(**params)
    scores_ = cross_val_score(model, X_train_sel, y_train,
                               cv=tscv, scoring='f1', n_jobs=-1)
    return scores_.mean()

study_xgb = optuna.create_study(direction='maximize',
                                 sampler=optuna.samplers.TPESampler(seed=SEED))
study_xgb.optimize(xgb_objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f'\n✅ Best XGB params (F1={study_xgb.best_value:.4f}):')
print(study_xgb.best_params)

In [ ]:
# ── 7.3 Train Tuned Final Model ──────────────────────────────────────────────
lgb_tuned = lgb.LGBMClassifier(**study_lgb.best_params)
xgb_tuned = xgb.XGBClassifier(**study_xgb.best_params)

lgb_tuned_m = eval_model('LGB_Tuned', lgb_tuned, X_train_sel, y_train, X_test_sel, y_test)
xgb_tuned_m = eval_model('XGB_Tuned', xgb_tuned, X_train_sel, y_train, X_test_sel, y_test)

# ── Tuned Voting ─────────────────────────────────────────────────────────────
voting_tuned = VotingClassifier(
    estimators=[
        ('lgb', lgb.LGBMClassifier(**study_lgb.best_params)),
        ('xgb', xgb.XGBClassifier(**study_xgb.best_params)),
        ('rf',  RandomForestClassifier(n_estimators=300, max_depth=8,
                                        min_samples_leaf=10, n_jobs=-1, random_state=SEED))
    ],
    voting='soft', weights=[2, 2, 1]
)
voting_tuned_m = eval_model('Voting_Tuned', voting_tuned,
                             X_train_sel, y_train, X_test_sel, y_test)

---
## 📈 Section 8 — Walk-Forward Validation

In [ ]:
# ── 8.1 Walk-Forward CV ──────────────────────────────────────────────────────
WF_SPLITS = 5
tscv_wf   = TimeSeriesSplit(n_splits=WF_SPLITS)

def walkforward_eval(name, model, X, y):
    fold_accs  = []
    fold_f1s   = []
    fold_conf  = []

    for fold, (train_idx, test_idx) in enumerate(tscv_wf.split(X)):
        X_tr = X[train_idx]; y_tr = y.iloc[train_idx]
        X_te = X[test_idx];  y_te = y.iloc[test_idx]

        model.fit(X_tr, y_tr)
        prob = model.predict_proba(X_te)[:, 1]
        pred = (prob >= 0.5).astype(int)

        acc = accuracy_score(y_te, pred)
        f1  = f1_score(y_te, pred, zero_division=0)

        conf_mask = (prob >= 0.70) | (prob <= 0.30)
        conf_acc  = accuracy_score(y_te[conf_mask], pred[conf_mask]) if conf_mask.sum() > 0 else 0

        fold_accs.append(acc)
        fold_f1s.append(f1)
        fold_conf.append(conf_acc)
        print(f'  Fold {fold+1}: Acc={acc:.4f}  F1={f1:.4f}  ConfAcc={conf_acc:.4f}')

    print(f'  ─── [{name}] Mean Acc={np.mean(fold_accs):.4f}±{np.std(fold_accs):.4f}'
          f'  F1={np.mean(fold_f1s):.4f}  ConfAcc={np.mean(fold_conf):.4f}\n')
    return fold_accs, fold_f1s, fold_conf

# Use fresh instances (refit required)
print('=== Walk-Forward: LGB Tuned ===')
wf_lgb = walkforward_eval('LGB_Tuned',
                           lgb.LGBMClassifier(**study_lgb.best_params),
                           X_train_sel, y_train)

print('=== Walk-Forward: XGB Tuned ===')
wf_xgb = walkforward_eval('XGB_Tuned',
                           xgb.XGBClassifier(**study_xgb.best_params),
                           X_train_sel, y_train)

In [ ]:
# ── 8.2 Walk-Forward Chart ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
folds = range(1, WF_SPLITS+1)

for ax, metric, lgb_vals, xgb_vals, title in zip(
    axes,
    ['Accuracy','F1','ConfAcc'],
    wf_lgb, wf_xgb,
    ['Accuracy per Fold','F1 Score per Fold','Confident Accuracy per Fold']
):
    ax.plot(folds, lgb_vals, 'o-', label='LGB', color='#00d4ff')
    ax.plot(folds, xgb_vals, 's-', label='XGB', color='#ff6b35')
    ax.set_title(title)
    ax.set_xlabel('Fold')
    ax.legend()
    ax.set_ylim(0.4, 1.0)
    ax.axhline(0.75, color='gray', lw=1, ls='--', label='0.75 target')

plt.tight_layout()
plt.savefig('walkforward.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 🧠 Section 9 — Deep Learning (CNN-BiLSTM)

In [ ]:
# ── 9.1 Try TensorFlow / Keras ──────────────────────────────────────────────
TF_AVAILABLE = False
try:
    import tensorflow as tf
    from tensorflow.keras.models import Model
    from tensorflow.keras.layers import (
        Input, Conv1D, MaxPooling1D, Bidirectional, LSTM,
        Dense, Dropout, BatchNormalization, GlobalAveragePooling1D
    )
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
    from tensorflow.keras.optimizers import Adam
    TF_AVAILABLE = True
    print(f'✅ TensorFlow {tf.__version__} tersedia')
except ImportError:
    print('⚠️  TensorFlow tidak tersedia — skip LSTM section')
    print('   Install: pip install tensorflow')

In [ ]:
if TF_AVAILABLE:
    # ── 9.2 Sequence Dataset ─────────────────────────────────────────────────
    SEQ_LEN    = 30
    BATCH_SIZE = 64
    EPOCHS     = 60

    def make_sequences(X_arr, y_arr, seq_len):
        Xs, ys = [], []
        for i in range(seq_len, len(X_arr)):
            Xs.append(X_arr[i-seq_len:i])
            ys.append(y_arr.iloc[i])
        return np.array(Xs), np.array(ys)

    X_all_sel = np.vstack([X_train_sel, X_test_sel])
    y_all_arr = pd.concat([y_train, y_test]).reset_index(drop=True)

    Xs, ys = make_sequences(X_all_sel, y_all_arr, SEQ_LEN)
    n_seq  = len(Xs)
    split  = int(n_seq * 0.80)

    Xs_tr, Xs_te = Xs[:split], Xs[split:]
    ys_tr, ys_te = ys[:split], ys[split:]

    print(f'Sequence shapes: train={Xs_tr.shape} test={Xs_te.shape}')

    # ── 9.3 Build CNN-BiLSTM ─────────────────────────────────────────────────
    tf.random.set_seed(SEED)

    inp = Input(shape=(SEQ_LEN, K_FEATURES))

    # CNN feature extractor
    x = Conv1D(64, kernel_size=3, activation='relu', padding='same')(inp)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.2)(x)

    x = Conv1D(128, kernel_size=3, activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.2)(x)

    # BiLSTM
    x = Bidirectional(LSTM(128, return_sequences=True))(x)
    x = Dropout(0.3)(x)
    x = Bidirectional(LSTM(64, return_sequences=False))(x)
    x = Dropout(0.3)(x)

    # Dense
    x = Dense(64, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    x = Dense(32, activation='relu')(x)
    out = Dense(1, activation='sigmoid')(x)

    lstm_model = Model(inp, out)
    lstm_model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
    )
    lstm_model.summary()

    # ── 9.4 Train ────────────────────────────────────────────────────────────
    callbacks = [
        EarlyStopping(patience=10, restore_best_weights=True, monitor='val_auc', mode='max'),
        ReduceLROnPlateau(patience=5, factor=0.5, min_lr=1e-6, monitor='val_loss')
    ]

    history = lstm_model.fit(
        Xs_tr, ys_tr,
        validation_data=(Xs_te, ys_te),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        class_weight={0: 1.0, 1: 1.0},
        verbose=1
    )

    # ── 9.5 Evaluate ─────────────────────────────────────────────────────────
    prob_lstm = lstm_model.predict(Xs_te).ravel()
    pred_lstm = (prob_lstm >= 0.5).astype(int)
    acc_lstm  = accuracy_score(ys_te, pred_lstm)
    auc_lstm  = roc_auc_score(ys_te, prob_lstm)
    f1_lstm   = f1_score(ys_te, pred_lstm, zero_division=0)
    cm        = (prob_lstm >= 0.70) | (prob_lstm <= 0.30)
    conf_lstm = accuracy_score(ys_te[cm], pred_lstm[cm]) if cm.sum() > 0 else 0

    RESULTS['CNN-BiLSTM'] = {
        'acc': acc_lstm, 'prec': precision_score(ys_te, pred_lstm, zero_division=0),
        'rec': recall_score(ys_te, pred_lstm, zero_division=0),
        'f1': f1_lstm, 'auc': auc_lstm,
        'conf_acc': conf_lstm, 'conf_cov': cm.mean()*100
    }

    print(f'\n[CNN-BiLSTM]')
    print(f'  Acc={acc_lstm:.4f}  F1={f1_lstm:.4f}  AUC={auc_lstm:.4f}  ConfAcc={conf_lstm:.4f}')

    # ── Training curves ───────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(history.history['accuracy'],    label='Train Acc', color='#00d4ff')
    axes[0].plot(history.history['val_accuracy'],label='Val Acc',   color='#ff6b35')
    axes[0].set_title('CNN-BiLSTM Accuracy'); axes[0].legend()

    axes[1].plot(history.history['loss'],    label='Train Loss', color='#00d4ff')
    axes[1].plot(history.history['val_loss'],label='Val Loss',   color='#ff6b35')
    axes[1].set_title('CNN-BiLSTM Loss'); axes[1].legend()
    plt.tight_layout()
    plt.savefig('lstm_training.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('TensorFlow tidak tersedia — skip CNN-BiLSTM')

---
## 🔍 Section 10 — SHAP Feature Importance

In [ ]:
import shap

# ── 10.1 SHAP untuk LGB Tuned ────────────────────────────────────────────────
lgb_final = lgb.LGBMClassifier(**study_lgb.best_params)
lgb_final.fit(X_train_sel, y_train)

explainer  = shap.TreeExplainer(lgb_final)
shap_vals  = explainer.shap_values(X_test_sel)

# shap_vals is list [class0, class1] for binary
sv = shap_vals[1] if isinstance(shap_vals, list) else shap_vals
feat_arr = np.array(selected_features)

print('SHAP Feature Importance (Mean |SHAP|):')
mean_shap = np.abs(sv).mean(axis=0)
shap_df   = pd.DataFrame({'feature': feat_arr, 'mean_shap': mean_shap})
shap_df   = shap_df.sort_values('mean_shap', ascending=False)

fig, ax = plt.subplots(figsize=(10, 12))
top30 = shap_df.head(30)
ax.barh(top30['feature'][::-1], top30['mean_shap'][::-1], color='#00d4ff')
ax.set_title('Top 30 Features by Mean |SHAP|')
ax.set_xlabel('Mean |SHAP value|')
plt.tight_layout()
plt.savefig('shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print(shap_df.head(20).to_string())

In [ ]:
# ── 10.2 SHAP Beeswarm ───────────────────────────────────────────────────────
shap_explanation = shap.Explanation(
    values=sv,
    base_values=explainer.expected_value[1] if isinstance(explainer.expected_value, list)
                else explainer.expected_value,
    data=X_test_sel,
    feature_names=selected_features
)

fig = plt.figure(figsize=(12, 10))
shap.plots.beeswarm(shap_explanation, max_display=20, show=False)
plt.title('SHAP Beeswarm — LGB Tuned')
plt.tight_layout()
plt.savefig('shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 💰 Section 11 — Signal Backtest

In [ ]:
# ── 11.1 Signal Backtest dengan model terbaik ────────────────────────────────
# Ambil probabilitas dari Voting Tuned (atau LGB Tuned)

# Refit Voting Tuned pada full train, evaluate on test
voting_tuned.fit(X_train_sel, y_train)
prob_test = voting_tuned.predict_proba(X_test_sel)[:, 1]

# Index waktu untuk test period
test_idx = df_clean.iloc[split_idx:].index

# Ambil OHLCV untuk test period
df_test_ohlcv = df_raw.loc[test_idx].copy()

# Sesuaikan panjang (mungkin beda karena NaN drops)
min_len = min(len(prob_test), len(df_test_ohlcv))
prob_test      = prob_test[:min_len]
df_test_ohlcv  = df_test_ohlcv.iloc[:min_len].copy()

df_test_ohlcv['prob']    = prob_test
df_test_ohlcv['signal']  = np.where(prob_test >= 0.65, 'BUY',
                             np.where(prob_test <= 0.35, 'SELL', 'WAIT'))

print(f'Signal distribution on test set:')
print(df_test_ohlcv['signal'].value_counts().to_string())

In [ ]:
# ── 11.2 Simple PnL Simulation ──────────────────────────────────────────────
INITIAL_CAPITAL = 10_000
RISK_PER_TRADE  = 0.01      # 1% per trade
RR_RATIO        = 3.0       # Target RR 1:3
ATR_SL_MULT     = 1.0       # SL = 1x ATR
SPREAD_USD      = 0.35      # per lot
PIP_USD         = 0.10      # per pip per 0.01 lot

def simulate_trades(df_sig, initial_capital=INITIAL_CAPITAL,
                    risk_pct=RISK_PER_TRADE, rr=RR_RATIO, atr_sl=ATR_SL_MULT):
    capital    = initial_capital
    trades     = []
    peak_cap   = capital
    max_dd     = 0
    
    # Simple ATR
    atr = (df_sig['High'] - df_sig['Low']).rolling(14).mean()

    i = 0
    while i < len(df_sig) - LOOKAHEAD - 1:
        row = df_sig.iloc[i]
        sig = row['signal']

        if sig == 'WAIT':
            i += 1
            continue

        entry    = row['Close']
        atr_val  = atr.iloc[i] if not np.isnan(atr.iloc[i]) else 1.0
        sl_dist  = atr_val * atr_sl
        tp_dist  = sl_dist * rr

        sl = entry - sl_dist if sig == 'BUY' else entry + sl_dist
        tp = entry + tp_dist if sig == 'BUY' else entry - tp_dist

        risk_usd = capital * risk_pct

        # Walk forward to find exit
        result = 'OPEN'
        exit_price = entry
        exit_i = i + 1

        for j in range(i+1, min(i+1+50, len(df_sig))):
            h = df_sig.iloc[j]['High']
            l = df_sig.iloc[j]['Low']

            if sig == 'BUY':
                if l <= sl:    result = 'LOSS'; exit_price = sl; exit_i = j; break
                if h >= tp:    result = 'WIN';  exit_price = tp; exit_i = j; break
            else:
                if h >= sl:    result = 'LOSS'; exit_price = sl; exit_i = j; break
                if l <= tp:    result = 'WIN';  exit_price = tp; exit_i = j; break

        if result == 'OPEN':
            exit_price = df_sig.iloc[min(i+50, len(df_sig)-1)]['Close']
            pnl_raw = (exit_price - entry) if sig == 'BUY' else (entry - exit_price)
            result  = 'WIN' if pnl_raw > 0 else 'LOSS'

        pnl_raw = (exit_price - entry) if sig == 'BUY' else (entry - exit_price)
        pnl_pct = pnl_raw / entry
        pnl_usd = risk_usd * (pnl_raw / sl_dist) - SPREAD_USD

        capital += pnl_usd
        peak_cap = max(peak_cap, capital)
        dd       = (peak_cap - capital) / peak_cap * 100
        max_dd   = max(max_dd, dd)

        trades.append({
            'open_time'  : df_sig.index[i],
            'close_time' : df_sig.index[exit_i],
            'direction'  : sig,
            'entry'      : round(entry, 2),
            'exit'       : round(exit_price, 2),
            'sl'         : round(sl, 2),
            'tp'         : round(tp, 2),
            'result'     : result,
            'pnl_usd'    : round(pnl_usd, 2),
            'capital'    : round(capital, 2),
            'drawdown'   : round(dd, 2)
        })

        i = exit_i + 1

    trades_df = pd.DataFrame(trades)

    if len(trades_df) == 0:
        return {'total_trades': 0}, pd.DataFrame()

    wins  = (trades_df['result'] == 'WIN').sum()
    total = len(trades_df)
    sum_win  = trades_df.loc[trades_df['result']=='WIN',  'pnl_usd'].sum()
    sum_loss = trades_df.loc[trades_df['result']=='LOSS', 'pnl_usd'].abs().sum()

    stats = {
        'total_trades'  : total,
        'wins'          : int(wins),
        'losses'        : int(total - wins),
        'win_rate'      : round(wins/total*100, 2),
        'total_pnl_usd' : round(trades_df['pnl_usd'].sum(), 2),
        'profit_factor' : round(sum_win / (sum_loss+1e-9), 3),
        'max_drawdown'  : round(max_dd, 2),
        'final_capital' : round(capital, 2),
        'return_pct'    : round((capital-initial_capital)/initial_capital*100, 2),
        'avg_win'       : round(trades_df.loc[trades_df['result']=='WIN','pnl_usd'].mean(), 2),
        'avg_loss'      : round(trades_df.loc[trades_df['result']=='LOSS','pnl_usd'].mean(), 2)
    }
    return stats, trades_df

bt_stats, bt_trades = simulate_trades(df_test_ohlcv)

print('\n💰 Backtest Results:')
for k, v in bt_stats.items():
    print(f'  {k:<20}: {v}')

In [ ]:
# ── 11.3 Equity Curve ───────────────────────────────────────────────────────
if len(bt_trades) > 0:
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=['Equity Curve', 'Drawdown (%)'],
                        row_heights=[0.7, 0.3])

    fig.add_trace(
        go.Scatter(x=bt_trades['close_time'], y=bt_trades['capital'],
                   mode='lines', name='Equity', line=dict(color='#00d4ff', width=2)),
        row=1, col=1
    )
    fig.add_hline(y=INITIAL_CAPITAL, line_dash='dash', line_color='gray',
                  row=1, col=1)

    fig.add_trace(
        go.Scatter(x=bt_trades['close_time'], y=-bt_trades['drawdown'],
                   mode='lines', fill='tozeroy',
                   name='Drawdown', line=dict(color='#ff4466', width=1)),
        row=2, col=1
    )

    fig.update_layout(
        title=f'Backtest | WinRate={bt_stats["win_rate"]}% | PF={bt_stats["profit_factor"]} | Return={bt_stats["return_pct"]}%',
        template='plotly_dark', height=600, showlegend=True
    )
    fig.write_html('equity_curve.html')
    fig.show()
    print('✅ equity_curve.html saved')
else:
    print('⚠️  No trades generated')

In [ ]:
# ── 11.4 Trade Distribution ──────────────────────────────────────────────────
if len(bt_trades) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # PnL distribution
    axes[0].hist(bt_trades['pnl_usd'], bins=40,
                 color=['#00ff88' if x > 0 else '#ff4466' for x in bt_trades['pnl_usd']],
                 edgecolor='none', alpha=0.8)
    axes[0].axvline(0, color='white', lw=1)
    axes[0].set_title('PnL Distribution (USD)')

    # Win/Loss by direction
    dir_res = bt_trades.groupby(['direction','result']).size().unstack(fill_value=0)
    dir_res.plot(kind='bar', ax=axes[1], color=['#ff4466','#00ff88'], edgecolor='none')
    axes[1].set_title('Win/Loss by Direction')
    axes[1].tick_params(axis='x', rotation=0)

    # Monthly returns
    if 'close_time' in bt_trades.columns:
        bt_trades['month'] = pd.to_datetime(bt_trades['close_time']).dt.to_period('M')
        monthly = bt_trades.groupby('month')['pnl_usd'].sum()
        colors  = ['#00ff88' if v > 0 else '#ff4466' for v in monthly.values]
        axes[2].bar(monthly.index.astype(str), monthly.values, color=colors)
        axes[2].set_title('Monthly PnL (USD)')
        axes[2].tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.savefig('trade_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()

---
## 📊 Section 12 — Model Comparison & ROC Curves

In [ ]:
# ── 12.1 Comparison Table ────────────────────────────────────────────────────
results_df = pd.DataFrame(RESULTS).T
results_df = results_df.round(4)
results_df = results_df.sort_values('conf_acc', ascending=False)
print('\n📊 Model Comparison:')
print(results_df.to_string())

In [ ]:
# ── 12.2 Visual Comparison ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

metrics = ['acc','f1','auc','conf_acc']
colors  = ['#00d4ff','#ff6b35','#c3f0ca','#f7971e']

x = np.arange(len(results_df))
w = 0.2

for i, (metric, color) in enumerate(zip(metrics, colors)):
    axes[0].bar(x + i*w, results_df[metric], width=w, label=metric.upper(), color=color)

axes[0].set_xticks(x + w*1.5)
axes[0].set_xticklabels(results_df.index, rotation=30, ha='right', fontsize=9)
axes[0].legend()
axes[0].set_title('Model Performance Comparison')
axes[0].axhline(0.75, color='white', lw=1, ls='--', alpha=0.5)
axes[0].set_ylim(0, 1.05)

# ROC Curves
axes[1].plot([0,1],[0,1], 'k--', lw=1)
model_roc = [
    ('LGB_Tuned',      lgb_tuned_m,     '#00d4ff'),
    ('XGB_Tuned',      xgb_tuned_m,     '#ff6b35'),
    ('Voting_Tuned',   voting_tuned_m,  '#c3f0ca'),
    ('Stacking',       stacking_m,      '#f7971e'),
]

for name, model, color in model_roc:
    if hasattr(model, 'predict_proba'):
        try:
            pp  = model.predict_proba(X_test_sel)[:, 1]
            fpr, tpr, _ = roc_curve(y_test, pp)
            auc_v = roc_auc_score(y_test, pp)
            axes[1].plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC={auc_v:.3f})')
        except Exception:
            pass

axes[1].set_title('ROC Curves')
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 12.3 Confusion Matrix — Best Model ──────────────────────────────────────
best_model_name = results_df.index[0]
print(f'Best model: {best_model_name}')

# Re-evaluate best on test
voting_tuned.fit(X_train_sel, y_train)
pp_best = voting_tuned.predict_proba(X_test_sel)[:, 1]
pred_best = (pp_best >= 0.5).astype(int)

cm = confusion_matrix(y_test, pred_best)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['SELL','BUY'], yticklabels=['SELL','BUY'])
axes[0].set_title(f'Confusion Matrix — {best_model_name}')

# Confidence calibration curve
bins   = np.linspace(0, 1, 11)
mid    = (bins[:-1] + bins[1:]) / 2
actual = []
for lo, hi in zip(bins[:-1], bins[1:]):
    mask = (pp_best >= lo) & (pp_best < hi)
    if mask.sum() > 0:
        actual.append(y_test[mask].mean())
    else:
        actual.append(np.nan)

axes[1].plot(mid, actual, 'o-', color='#00d4ff', label='Model')
axes[1].plot([0,1],[0,1], 'k--', lw=1, label='Perfect calibration')
axes[1].set_title('Calibration Curve')
axes[1].set_xlabel('Predicted Probability')
axes[1].set_ylabel('Actual Win Rate')
axes[1].legend()

plt.tight_layout()
plt.savefig('confusion_calibration.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nClassification Report:')
print(classification_report(y_test, pred_best, target_names=['SELL','BUY']))

---
## 💾 Section 13 — Export Model Terbaik

In [ ]:
# ── 13.1 Train final model on FULL data ─────────────────────────────────────
X_full_s   = scaler.fit_transform(X_all)
X_full_sel = selector.fit_transform(X_full_s, y_all)

final_voting = VotingClassifier(
    estimators=[
        ('lgb', lgb.LGBMClassifier(**study_lgb.best_params)),
        ('xgb', xgb.XGBClassifier(**study_xgb.best_params)),
        ('rf',  RandomForestClassifier(n_estimators=300, max_depth=8,
                                        min_samples_leaf=10, n_jobs=-1, random_state=SEED))
    ],
    voting='soft', weights=[2, 2, 1]
)
final_voting.fit(X_full_sel, y_all)
print('✅ Final model trained on full data')

# ── 13.2 Save ────────────────────────────────────────────────────────────────
SAVE_DIR = ROOT / 'models'
SAVE_DIR.mkdir(exist_ok=True)

bundle = {
    'model'    : final_voting,
    'scaler'   : scaler,
    'selector' : selector,
    'features' : list(X_all.columns),
    'selected' : selected_features,
    'metrics'  : RESULTS
}
joblib.dump(bundle, SAVE_DIR / 'ml_XAUUSD_5m_v2.joblib', compress=3)

# Metadata
meta = {
    'trained_symbol'    : 'XAUUSD',
    'trained_timeframe' : '5m',
    'trained_n_candles' : int(len(df_clean)),
    'last_candle_time'  : str(df_clean.index[-1]),
    'test_accuracy'     : float(RESULTS.get(best_model_name, {}).get('acc', 0)),
    'conf_accuracy'     : float(RESULTS.get(best_model_name, {}).get('conf_acc', 0)),
    'f1_score'          : float(RESULTS.get(best_model_name, {}).get('f1', 0)),
    'auc'               : float(RESULTS.get(best_model_name, {}).get('auc', 0)),
    'n_features'        : K_FEATURES,
    'lookahead'         : LOOKAHEAD,
    'threshold_pct'     : THRESHOLD_PCT,
    'lgb_best_params'   : study_lgb.best_params,
    'xgb_best_params'   : study_xgb.best_params,
    'backtest'          : bt_stats
}

with open(SAVE_DIR / 'ml_XAUUSD_5m_v2_meta.json', 'w') as f:
    json.dump(meta, f, indent=2, default=str)

print(f'\n✅ Model saved to: {SAVE_DIR / "ml_XAUUSD_5m_v2.joblib"}')
print(f'✅ Meta  saved to: {SAVE_DIR / "ml_XAUUSD_5m_v2_meta.json"}')
print('\n📌 Final Metrics:')
print(f'   Accuracy     : {meta["test_accuracy"]*100:.2f}%')
print(f'   Conf Accuracy: {meta["conf_accuracy"]*100:.2f}%')
print(f'   F1 Score     : {meta["f1_score"]*100:.2f}%')
print(f'   AUC          : {meta["auc"]:.4f}')

In [ ]:
# ── 13.3 Summary Dashboard ──────────────────────────────────────────────────
print('=' * 60)
print('  XAUUSD ML ANALYSIS — FINAL SUMMARY')
print('=' * 60)
print(f'  Data          : {len(df_raw):,} candles (5m XAUUSD)')
print(f'  Labeled       : {len(df_clean):,} (sideways removed)')
print(f'  Features      : {K_FEATURES} selected from {len(X_all.columns)}')
print(f'  Lookahead     : {LOOKAHEAD} candles @ {THRESHOLD_PCT*100:.2f}% threshold')
print()
print('  Model Rankings (by Confident Accuracy):')
for i, (name, row) in enumerate(results_df.iterrows()):
    print(f'  {i+1}. {name:<25} Acc={row["acc"]*100:.1f}%  '
          f'ConfAcc={row["conf_acc"]*100:.1f}%  F1={row["f1"]*100:.1f}%  AUC={row["auc"]:.3f}')
print()
print('  Backtest Results:')
for k, v in bt_stats.items():
    print(f'  {k:<20}: {v}')
print()
print('  Top 10 Features (SHAP):')
for i, (feat, score) in enumerate(shap_df.head(10).iterrows()):
    if isinstance(feat, int):
        feat = shap_df.iloc[i]['feature']
    print(f'  {i+1:2}. {shap_df.iloc[i]["feature"]:<30} {shap_df.iloc[i]["mean_shap"]:.4f}')
print('=' * 60)